## Data preparation

We intend to use spectral embedding on this dataset to convert it into a set of continuous latent variables.  
Often this algorithm requires a preliminary unsupervised learning to identify related data points (typically using Random Forests) but that is not necessary here as we already have our data in the form of user-to-song associations which can be viewed as edges in a graph.

However we still need to convert the source table to a sparse matrix, which we will implement in this notebook.

Import the essential libraries that we need

In [1]:
import os

Set the working directory to the project root to simplify access to the data and other Python modules

In [2]:
current_dir = os.getcwd()
project_directory, notebooks_subdirectory = os.path.split(current_dir)
if notebooks_subdirectory == "notebooks":
    os.chdir(project_directory)
    print(f"Changed working directory to project root: {project_directory}")
else:
    project_directory = current_dir
# Confirm the new current directory
os.getcwd()

Changed working directory to project root: c:\Users\dave4\vscode-projects\spotify-playlists


'c:\\Users\\dave4\\vscode-projects\\spotify-playlists'

Also add our local modules to the import path

In [3]:
#data_directory = os.path.join(project_directory, "data")
if project_directory not in os.sys.path:
    os.sys.path.append(project_directory)

Set up paths to all the relevant files and directories

In [4]:
from pipeline import Pipeline
pipeline = Pipeline(project_directory)

Test that the CSV parsing is working, as well as our song title cleaning
(some song tiles in the source data contain quotes and commas which are not properly escaped)

Here we display the first 10 entries, plus a few others that are known to contain special characters

In [5]:
for row_num, row in enumerate(pipeline.source_view.stream(pipeline)):
    if row_num < 10 or row_num in {228,299,5109,7288}:
        print(row_num, row, len(row))

print(f"Total rows found: {row_num}")

0 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello', '(The Angels Wanna Wear My) Red Shoes', 'HARD ROCK 2010'] 4
1 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello & The Attractions', "(What's So Funny 'Bout) Peace, Love And Understanding", 'HARD ROCK 2010'] 4
2 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Tiffany Page', '7 Years Too Late', 'HARD ROCK 2010'] 4
3 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello & The Attractions', 'Accidents Will Happen', 'HARD ROCK 2010'] 4
4 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello', 'Alison', 'HARD ROCK 2010'] 4
5 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Lissie', 'All Be Okay', 'HARD ROCK 2010'] 4
6 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Paul McCartney', 'Band On The Run', 'HARD ROCK 2010'] 4
7 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Joe Echo', 'Beautiful', 'HARD ROCK 2010'] 4
8 ['9cc0cfd4d7d7885102480dd99e7a90d6', 'Paul McCartney', 'Blackbird - Live at CitiField, NYC - Digital Audio', 'HARD ROCK 2010'] 4
9 ['9cc0cfd4d7d7885102480dd99e7a90

This dataset is fairly large, more than a gigabyte to hold the whole CSV in memory so we stream it one row at a time and construct a label encoding.

After that, we have just a pair of integers for each entry, requiring far less space.

In [6]:
pipeline.encode_listens()

Encoded listens, users, and songs already exist. Skipping encoding step.


DatasetStats(num_users=15918, num_songs=2824004, num_listens=11413972)

We will still struggle to perform a full matrix factorisation on this many entries, so further restrict the dataset to songs with a moderately-large number of unique listeners. A limit of 200 works well. This could possibly be tuned further if we can find a way to process a larger matrix.

At the same time, apply a pseudo-random permutation to the user IDs, so we can then take a test/train split of the user set by splitting the numeric range of IDs.

This could otherwise be biased because the original user IDs were assigned in order of their first appearance in the original dataset.

In [7]:
pipeline.support_threshold = 6  # 6 works reliably on the author's machine; 5 can work if other memory-intensive processes are not running. Adjust as needed based on your system's capabilities.
pipeline.apply_threshold()  # Allow 1-2 minutes for this operation

ThresholdStats(num_users=15689, num_songs=321884, num_listens=7556032, min_listens=6)

In [8]:
pipeline.status()

,view,files_present,file_paths
0,SourceView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
1,UserLabelsView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
2,SongLabelsView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
3,FullListensView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
4,StatsView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
5,FilteredListensView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
6,FilteredSongsView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
7,FilteredUsersView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
8,ThresholdStatsView,True,c:\Users\dave4\vscode-projects\spotify-playlis...
9,UserEmbeddingsView,False,c:\Users\dave4\vscode-projects\spotify-playlis...


In [9]:
pipeline.build_embeddings()  # Allow at least 4-6 minutes for this operation